In [ ]:
"""
TNE Research Analysis: Australian Higher Education Enrolment Plots (2020-2024)
================================================================================

Generates a series of plots that connect the Department of Education enrolment
data to the TNE (Transnational Education) research findings:

  1. ANU vs. Go8 peers — total enrolment trajectory (2020-2024)
  2. ANU vs. comparable Go8 (UWA) — head-to-head benchmarking
  3. ANU's 2024 field-of-education mix (where TNE programmes could fit)
  4. Top 10 Australian universities by 2024 enrolment — scale comparison
  5. Australian HE growth rates 2020-2024 — winners and losers
  6. ANU's enrolment vs. NOSC shortfall context (TNE opportunity sizing)

Data source:
    Department of Education, Higher Education Statistics Collection,
    "Perturbed Student Enrolments Pivot Table 2024" (perturbed for disclosure
    control - not exact figures but reliable for trend analysis)

Author: Strategic Analysis Team
Date:   April 2026
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

# -----------------------------------------------------------------------------
# Plot styling - consistent palette aligned with prior TNE report visuals
# -----------------------------------------------------------------------------
ANU_COLOR     = "#1B3A5C"   # ANU navy
HIGHLIGHT     = "#B45309"   # warm accent (used in prior key-message highlights)
GO8_COLORS    = ["#2C5282", "#4A6FA5", "#7C9CC0", "#A4C2E0", "#3266AD",
                 "#534AB7", "#7C3AED"]  # cool palette for peer Go8 universities
NEUTRAL_GREY  = "#6B7280"
LIGHT_GREY    = "#E5E7EB"
SUCCESS_GREEN = "#059669"
DANGER_RED    = "#B91C1C"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":         10,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   "#1B3A5C",
    "axes.labelsize":    10,
    "axes.labelcolor":   "#374151",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.edgecolor":    "#9CA3AF",
    "axes.linewidth":    0.8,
    "xtick.color":       "#374151",
    "ytick.color":       "#374151",
    "grid.color":        "#E5E7EB",
    "grid.linewidth":    0.5,
    "legend.frameon":    False,
    "legend.fontsize":   9,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
})

# -----------------------------------------------------------------------------
# Load and clean the data
# -----------------------------------------------------------------------------
SRC = "/mnt/user-data/uploads/Perturbed_Student_Enrolments_Pivot_Table_2024.xlsx"
OUT = Path("/mnt/user-data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

df_inst = pd.read_excel(SRC, sheet_name="Pivot", skiprows=24)
df_inst.columns = ["State", "Institution", 2020, 2021, 2022, 2023, 2024]
df_inst["State"] = df_inst["State"].ffill()
inst = df_inst[df_inst["Institution"].notna()].copy()
inst = inst[inst["Institution"] != "Grand Total"].reset_index(drop=True)

YEARS = [2020, 2021, 2022, 2023, 2024]

# Field of Education sheet (2024 only) - header row at index 23
df_foe = pd.read_excel(SRC, sheet_name="Pivot_BFOE", skiprows=23)
df_foe["State"] = df_foe["State"].ffill()
foe = df_foe[df_foe["Institution"].notna() & (df_foe["Institution"] != "Grand Total")].copy()


# -----------------------------------------------------------------------------
# PLOT 1: ANU vs. Go8 peers — total enrolment trajectory (2020-2024)
# -----------------------------------------------------------------------------
print("Generating Plot 1: ANU vs. Go8 enrolment trajectories...")

go8 = [
    "The Australian National University",
    "The University of Melbourne",
    "Monash University",
    "The University of Sydney",
    "University of New South Wales",
    "The University of Queensland",
    "The University of Western Australia",
    "The University of Adelaide",
]
go8_df = inst[inst["Institution"].isin(go8)].set_index("Institution")[YEARS]

fig, ax = plt.subplots(figsize=(11, 6.5))

# Plot peers in muted colours, then ANU in bold navy on top
peers = [u for u in go8 if u != "The Australian National University"]
for i, u in enumerate(peers):
    if u in go8_df.index:
        vals = go8_df.loc[u].values
        ax.plot(YEARS, vals, marker="o", markersize=5, linewidth=1.6,
                color=GO8_COLORS[i % len(GO8_COLORS)], alpha=0.55,
                label=u.replace("The University of ", "").replace("The ", "").replace("University", "Uni"))

# ANU on top, bold
anu_vals = go8_df.loc["The Australian National University"].values
ax.plot(YEARS, anu_vals, marker="o", markersize=8, linewidth=3.2,
        color=ANU_COLOR, label="ANU", zorder=10)

# Annotate ANU's start and end values
ax.annotate(f"{anu_vals[0]:,}", xy=(YEARS[0], anu_vals[0]),
            xytext=(-12, 8), textcoords="offset points",
            fontsize=9, color=ANU_COLOR, fontweight="bold", ha="right")
ax.annotate(f"{anu_vals[-1]:,}\n(\u22125.7%)", xy=(YEARS[-1], anu_vals[-1]),
            xytext=(8, -2), textcoords="offset points",
            fontsize=9, color=ANU_COLOR, fontweight="bold", ha="left")

ax.set_title("ANU has shrunk while most Go8 peers have grown\nTotal enrolments by Go8 university, 2020\u20132024", loc="left", pad=14)
ax.set_xlabel("Year")
ax.set_ylabel("Total student enrolments")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
ax.set_xticks(YEARS)
ax.grid(True, axis="y", alpha=0.5)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))

fig.text(0.01, 0.01,
         "Source: Australian Department of Education, Higher Education Statistics Collection 2024 (perturbed pivot data)",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot1_anu_vs_go8_trajectory.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# PLOT 2: ANU vs. UWA - head-to-head Go8 size-comparable benchmark
# -----------------------------------------------------------------------------
print("Generating Plot 2: ANU vs. UWA head-to-head...")

fig, ax = plt.subplots(figsize=(10, 6))

anu = inst[inst["Institution"] == "The Australian National University"][YEARS].values.flatten()
uwa = inst[inst["Institution"] == "The University of Western Australia"][YEARS].values.flatten()

ax.plot(YEARS, anu, marker="o", markersize=10, linewidth=3.2,
        color=ANU_COLOR, label="ANU", zorder=5)
ax.plot(YEARS, uwa, marker="s", markersize=10, linewidth=3.2,
        color=HIGHLIGHT, label="UWA (closest Go8 size comparator)", zorder=5)

# Fill between to show widening gap
ax.fill_between(YEARS, anu, uwa, where=(uwa >= anu),
                color=HIGHLIGHT, alpha=0.08, interpolate=True)

# Annotations
for x, y in zip(YEARS, anu):
    ax.annotate(f"{y:,}", xy=(x, y), xytext=(0, -16), textcoords="offset points",
                ha="center", fontsize=8.5, color=ANU_COLOR, fontweight="bold")
for x, y in zip(YEARS, uwa):
    ax.annotate(f"{y:,}", xy=(x, y), xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=8.5, color=HIGHLIGHT, fontweight="bold")

# Growth callouts
anu_growth = (anu[-1] - anu[0]) / anu[0] * 100
uwa_growth = (uwa[-1] - uwa[0]) / uwa[0] * 100
ax.text(0.02, 0.97,
        f"ANU 2020\u20132024: {anu_growth:+.1f}%\nUWA 2020\u20132024: {uwa_growth:+.1f}%",
        transform=ax.transAxes, fontsize=10, va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.5", fc="#FEF3C7", ec="#B45309", lw=1))

ax.set_title("UWA is now ahead of ANU in total enrolments\u2014and is moving on TNE first\nANU vs. UWA total enrolments, 2020\u20132024", loc="left", pad=14)
ax.set_xlabel("Year")
ax.set_ylabel("Total student enrolments")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
ax.set_xticks(YEARS)
ax.grid(True, axis="y", alpha=0.5)
ax.legend(loc="lower right")

fig.text(0.01, 0.01,
         "Source: Australian Department of Education, Higher Education Statistics Collection 2024. UWA approved for India branch campuses (UGC, June 2025); ANU yet to announce TNE entry.",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot2_anu_vs_uwa_headtohead.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# PLOT 3: ANU's 2024 field-of-education mix (TNE programme opportunity)
# -----------------------------------------------------------------------------
print("Generating Plot 3: ANU 2024 Field of Education mix...")

anu_foe = foe[foe["Institution"] == "The Australian National University"].iloc[0]
foe_categories = [c for c in foe.columns if c not in ["State", "Institution", "Total Enrolments"]]
anu_foe_values = {cat: anu_foe[cat] for cat in foe_categories if pd.notna(anu_foe[cat]) and anu_foe[cat] > 0}

# Sort and prep
sorted_foe = sorted(anu_foe_values.items(), key=lambda x: x[1], reverse=True)
labels  = [k for k, _ in sorted_foe]
values  = [v for _, v in sorted_foe]
total   = sum(values)
pcts    = [v / total * 100 for v in values]

# Annotate which fields align with TNE opportunity
TNE_ALIGNED = {
    "Management and Commerce":   ("\u2605 TNE-aligned (top demand from India/Vietnam)", SUCCESS_GREEN),
    "Information Technology":    ("\u2605 TNE-aligned (top demand from India/Vietnam)", SUCCESS_GREEN),
    "Society and Culture":       ("\u25CF ANU policy/IR strength (differentiated TNE niche)", HIGHLIGHT),
    "Engineering and Related Technologies": ("\u25CB Possible TNE expansion", "#7C9CC0"),
    "Natural and Physical Sciences": ("\u25CB Strong subject ranking; possible TNE", "#7C9CC0"),
}

fig, ax = plt.subplots(figsize=(11, 7))

bar_colors = []
for lab in labels:
    if lab in TNE_ALIGNED:
        bar_colors.append(TNE_ALIGNED[lab][1])
    else:
        bar_colors.append(LIGHT_GREY)

bars = ax.barh(labels, values, color=bar_colors, edgecolor="white", linewidth=0.8)

# Annotate values and TNE flags
for bar, v, p, lab in zip(bars, values, pcts, labels):
    ax.text(bar.get_width() + 80, bar.get_y() + bar.get_height()/2,
            f"{int(v):,} ({p:.1f}%)", va="center", fontsize=9, color="#374151")
    if lab in TNE_ALIGNED:
        flag, _ = TNE_ALIGNED[lab]
        ax.text(60, bar.get_y() + bar.get_height()/2 - 0.2,
                flag, va="center", fontsize=8, color="white", fontweight="bold",
                style="italic")

ax.invert_yaxis()
ax.set_title("ANU\u2019s strongest fields are precisely where TNE demand is concentrated\nANU 2024 enrolments by Broad Field of Education", loc="left", pad=14)
ax.set_xlabel("Number of students enrolled")
ax.set_xlim(0, max(values) * 1.18)
ax.grid(True, axis="x", alpha=0.5)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=SUCCESS_GREEN, label="TNE-aligned (high India/Vietnam demand)"),
    Patch(facecolor=HIGHLIGHT,     label="ANU differentiated TNE niche (policy/IR/society)"),
    Patch(facecolor="#7C9CC0",     label="Possible TNE expansion"),
    Patch(facecolor=LIGHT_GREY,    label="Other fields"),
]
ax.legend(handles=legend_elements, loc="lower right")

fig.text(0.01, 0.01,
         "Source: Australian Department of Education, Higher Education Statistics Collection 2024 (Pivot_BFOE sheet, perturbed). " +
         "TNE alignment based on Indian/Vietnamese student preferences in Australia onshore (PRISMS, 2025).",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot3_anu_field_of_education_mix.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# PLOT 4: Top 10 Australian universities by 2024 enrolment (real unis only)
# -----------------------------------------------------------------------------
print("Generating Plot 4: Top 10 by 2024 enrolment...")

# Filter to real universities only (exclude aggregators and non-uni providers)
def is_real_uni_for_top10(name):
    if pd.isna(name): return False
    n = str(name)
    excludes = ["Non-University", "Private Universities", "Multi-State"]
    return not any(x in n for x in excludes)

real_unis = inst[inst["Institution"].apply(is_real_uni_for_top10)].copy()
top10 = real_unis.nlargest(10, 2024)[["Institution", 2024]].copy()
top10["short"] = (top10["Institution"]
                  .str.replace("The University of ", "")
                  .str.replace("The ", "")
                  .str.replace(" University", "")
                  .str.replace("Australian National", "ANU")
                  .str.replace(" Australia", "")
                  .str.replace("of Technology", "Technology")
                  .str.replace(" Sydney", " Sydney"))

# Find ANU's overall rank among real unis
anu_rank_overall = (real_unis.sort_values(2024, ascending=False)
                    .reset_index(drop=True))
anu_overall_rank = anu_rank_overall[anu_rank_overall["Institution"]
                                    == "The Australian National University"].index[0] + 1
anu_2024_val = inst[inst["Institution"] == "The Australian National University"][2024].values[0]

fig, ax = plt.subplots(figsize=(11, 6.8))

bar_colors = []
for s in top10["short"]:
    if "ANU" in s:
        bar_colors.append(ANU_COLOR)
    elif "Western Australia" in s or s == "Western":
        bar_colors.append(HIGHLIGHT)
    else:
        bar_colors.append("#A4C2E0")

bars = ax.bar(top10["short"], top10[2024], color=bar_colors, edgecolor="white", linewidth=1)

for bar, v in zip(bars, top10[2024]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1200,
            f"{int(v/1000)}k", ha="center", fontsize=9, color="#374151", fontweight="bold")

# Set y-limit higher to give room for the annotation
ax.set_ylim(0, max(top10[2024]) * 1.30)

# Annotation positioned above bars in upper-right space
ax.text(0.97, 0.97,
        f"ANU is ranked #{anu_overall_rank} by 2024 enrolment ({int(anu_2024_val):,} students)\n"
        "\u2014 below the top 10 by scale, but #32 in QS World 2026.\n"
        "This scale gap structurally limits ranking indicators\n"
        "that reward size (e.g. faculty-student ratio).",
        transform=ax.transAxes, fontsize=9.5, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.6", fc="#FEF3C7", ec="#B45309", lw=1))

ax.set_title("ANU\u2019s small scale is a structural ranking disadvantage\nTop 10 Australian universities by 2024 total enrolment", loc="left", pad=14)
ax.set_ylabel("Total student enrolments (2024)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
plt.xticks(rotation=30, ha="right")
ax.grid(True, axis="y", alpha=0.5)

fig.text(0.01, 0.01,
         "Source: Australian Department of Education, Higher Education Statistics Collection 2024 (perturbed pivot). Excludes non-university provider aggregates.",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot4_top10_2024_enrolment.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# PLOT 5: Australian HE growth rates 2020-2024 — winners and losers
# -----------------------------------------------------------------------------
print("Generating Plot 5: Growth rates 2020-2024...")

# Filter to universities (drop "Non-University..." aggregates and small ones)
exclude_terms = ["Non-University", "Private Universities", "Carnegie Mellon",
                 "University of Divinity", "Avondale", "Batchelor",
                 "Multi-State"]
def is_real_uni(name):
    if pd.isna(name): return False
    name_str = str(name)
    for t in exclude_terms:
        if t in name_str:
            return False
    return True

unis = inst[inst["Institution"].apply(is_real_uni)].copy()
unis["growth"] = (unis[2024] - unis[2020]) / unis[2020] * 100
unis = unis.sort_values("growth", ascending=True)
unis["short"] = unis["Institution"].str.replace("The University of ", "").str.replace("The ", "").str.replace(" University", "").str.replace("Australian National", "ANU").str.replace(" Australia", "")

fig, ax = plt.subplots(figsize=(10, 9))

bar_colors = []
for _, row in unis.iterrows():
    if "ANU" in row["short"]:
        bar_colors.append(ANU_COLOR)
    elif row["growth"] >= 0:
        bar_colors.append(SUCCESS_GREEN)
    else:
        bar_colors.append(DANGER_RED)
    # Highlight UWA
    if "Western Australia" in row["Institution"]:
        bar_colors[-1] = HIGHLIGHT

bars = ax.barh(unis["short"], unis["growth"], color=bar_colors, edgecolor="white", linewidth=0.6)

for bar, g in zip(bars, unis["growth"]):
    x = bar.get_width()
    ha = "left" if x >= 0 else "right"
    offset = 0.4 if x >= 0 else -0.4
    ax.text(x + offset, bar.get_y() + bar.get_height()/2,
            f"{g:+.1f}%", va="center", ha=ha, fontsize=8, color="#374151")

ax.axvline(0, color="#374151", linewidth=0.8)
ax.set_title("ANU declined while comparable Go8 peers grew\nTotal enrolment growth, 2020\u20132024, Australian universities", loc="left", pad=14)
ax.set_xlabel("Total enrolment change, 2020\u20132024 (%)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(True, axis="x", alpha=0.5)

# Legend
legend_elements = [
    Patch(facecolor=ANU_COLOR,     label="ANU"),
    Patch(facecolor=HIGHLIGHT,     label="UWA (closest Go8 size comparator)"),
    Patch(facecolor=SUCCESS_GREEN, label="Other universities (positive growth)"),
    Patch(facecolor=DANGER_RED,    label="Other universities (negative growth)"),
]
ax.legend(handles=legend_elements, loc="lower right")

fig.text(0.01, 0.01,
         "Source: Australian Department of Education, Higher Education Statistics Collection 2024 (perturbed pivot). Excludes non-university providers and aggregates.",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot5_growth_2020_2024.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# PLOT 6: ANU enrolment trajectory + TNE opportunity sizing (counterfactual)
# -----------------------------------------------------------------------------
print("Generating Plot 6: ANU + TNE counterfactual...")

fig, ax = plt.subplots(figsize=(12, 7))

# Historical ANU enrolment (2020-2024)
ax.plot(YEARS, anu, marker="o", markersize=10, linewidth=3.2,
        color=ANU_COLOR, label="ANU actual (2020\u20132024, DoE data)", zorder=5)

# Bridge from 2024 actual into projection - start projection at 2024 end value
proj_years = list(range(2024, 2031))   # 2024..2030
flat_baseline = [anu[-1]] * len(proj_years)
# TNE adds: Year 0=0, Y1=300/350, Y2=700/800, Y3=1150/1300, Y4=1700/1900, Y5=2000/2500, Y6=continued
tne_low  = [0,   0, 300,  700, 1150, 1700, 2000]
tne_high = [0,   0, 350,  800, 1300, 1900, 2500]

# Without-TNE baseline (flat from 2024)
ax.plot(proj_years, flat_baseline, marker="o", markersize=7, linewidth=2.4,
        color=NEUTRAL_GREY, linestyle="--",
        label="ANU without TNE (flat baseline projection)", zorder=4)

# With-TNE high-case
with_tne_high = [b + h for b, h in zip(flat_baseline, tne_high)]
ax.plot(proj_years, with_tne_high, marker="^", markersize=8, linewidth=2.8,
        color=SUCCESS_GREEN, linestyle="-",
        label="ANU + TNE (high case projection)", zorder=4)

# Projection range fill
with_tne_low = [b + l for b, l in zip(flat_baseline, tne_low)]
ax.fill_between(proj_years, with_tne_low, with_tne_high,
                color=SUCCESS_GREEN, alpha=0.18, label="TNE projection range")

# Vertical separator between historical and projection - aligned with 2024
ax.axvline(2024, color=NEUTRAL_GREY, linewidth=1, linestyle=":", alpha=0.6)
ax.text(2024.05, 27500, "  Projection \u2192", fontsize=9, color=NEUTRAL_GREY,
        style="italic", ha="left")
ax.text(2023.95, 27500, "Historical  ", fontsize=9, color=NEUTRAL_GREY,
        style="italic", ha="right")

# Annotate Year 5 TNE upside (which is 2029, the 5th projection year after 2024)
year5_x = 2029
year5_y = with_tne_high[5]   # index 5 = 2029
ax.annotate(
    "Year 5 TNE upside:\n+2,000\u20132,500 students\n(\u2248A$30\u201350M revenue)\nNOSC-exempt revenue",
    xy=(year5_x, year5_y),
    xytext=(2025.5, 28800),
    fontsize=10, color=SUCCESS_GREEN, fontweight="bold", ha="left",
    bbox=dict(boxstyle="round,pad=0.5", fc="#ECFDF5", ec=SUCCESS_GREEN, lw=1.2),
    arrowprops=dict(arrowstyle="->", color=SUCCESS_GREEN, lw=1.5))

# Annotate the 2024 actual point
ax.annotate(f"2024 actual: {int(anu[-1]):,}",
            xy=(2024, anu[-1]),
            xytext=(2022, 21500),
            fontsize=9, color=ANU_COLOR, fontweight="bold",
            ha="center",
            arrowprops=dict(arrowstyle="->", color=ANU_COLOR, lw=1.2, alpha=0.7))

ax.set_title("TNE could add 9\u201311% to ANU\u2019s student base by 2030\nANU enrolment 2020\u20132024 (actual) and 2024\u20132030 with TNE (projected)",
             loc="left", pad=14)
ax.set_xlabel("Year")
ax.set_ylabel("Total student enrolments")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
ax.set_xticks(YEARS + [2025, 2026, 2027, 2028, 2029, 2030])
ax.set_ylim(20500, 30500)
ax.grid(True, axis="y", alpha=0.5)
ax.legend(loc="upper left", framealpha=0.95)

fig.text(0.01, 0.01,
         "Sources: Historical \u2014 Australian Department of Education 2024. Projections \u2014 calculated from peer benchmarks " +
         "(Monash Malaysia: 11k+ students; Deakin GIFT City: opened 2024; WSU\u2013UEH Vietnam: 3,000+ graduates over a decade).",
         fontsize=8, color=NEUTRAL_GREY, style="italic")

plt.tight_layout()
plt.savefig(OUT / "plot6_tne_counterfactual.png", dpi=200, bbox_inches="tight")
plt.close()


# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("All 6 plots generated successfully:")
for i, name in enumerate([
    "plot1_anu_vs_go8_trajectory.png",
    "plot2_anu_vs_uwa_headtohead.png",
    "plot3_anu_field_of_education_mix.png",
    "plot4_top10_2024_enrolment.png",
    "plot5_growth_2020_2024.png",
    "plot6_tne_counterfactual.png",
], 1):
    print(f"  Plot {i}: {OUT / name}")
print("="*70)

# Print the key numbers for traceability
print("\nKey data points underpinning the plots:")
print(f"  ANU 2020 enrolment: {int(anu[0]):,}")
print(f"  ANU 2024 enrolment: {int(anu[-1]):,}")
print(f"  ANU 2020-2024 change: {(anu[-1]-anu[0])/anu[0]*100:+.1f}%")
print(f"  UWA 2020 enrolment: {int(uwa[0]):,}")
print(f"  UWA 2024 enrolment: {int(uwa[-1]):,}")
print(f"  UWA 2020-2024 change: {(uwa[-1]-uwa[0])/uwa[0]*100:+.1f}%")
print(f"\n  ANU Top FOE 2024:")
for k, v in sorted_foe[:5]:
    print(f"    {k}: {int(v):,}")